# Customer Retention Intelligence System

This notebook documents the final Customer Retention Intelligence System pipeline: data import, EDA, feature engineering, model comparison, customer segmentation, retention scoring, and explainability. The dataset is the Iranian Churn dataset from the UCI Machine Learning Repository.

## System ML Stack

This notebook follows the final repository pipeline, not the early churn baseline.

- Churn model candidates: Logistic Regression, Random Forest, XGBoost, and LightGBM.
- The best churn model is saved to `artifacts/iranian_churn_model.joblib`.
- Customer segmentation: KMeans.
- Retention decision layer: churn probability, customer value tier, service issue signals, priority score, and recommended action.
- n8n integration: `predict.py` accepts customer JSON and returns prediction, segment, and recommended action fields.

In [ ]:
system_components = {
    "classification_models": [
        "logistic_regression",
        "random_forest",
        "xgboost",
        "lightgbm",
    ],
    "segmentation_model": "kmeans",
    "decision_layer": [
        "risk_level",
        "customer_value_tier",
        "retention_priority_score",
        "recommended_action",
    ],
}

system_components

## 1. Import Library

In [ ]:
from pathlib import Path
import importlib.util
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
BASE_DIR = Path("..").resolve()
DATA_PATH = BASE_DIR / "data" / "raw" / "Customer Churn.csv"
ARTIFACT_DIR = BASE_DIR / "artifacts"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", palette="Set2")

## 2. Import Data

In [ ]:
def clean_column_name(column):
    return (
        str(column)
        .strip()
        .lower()
        .replace("  ", " ")
        .replace(" ", "_")
    )


def clean_dataframe_columns(data):
    data = data.copy()
    data.columns = [clean_column_name(column) for column in data.columns]
    return data


df_raw = pd.read_csv(DATA_PATH)
df = clean_dataframe_columns(df_raw)

display(df.head())
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

In [ ]:
target_col = "churn"
raw_feature_columns = [column for column in df.columns if column != target_col]

print("Dataset source: UCI Iranian Churn Dataset")
print("URL: https://archive.ics.uci.edu/dataset/563/iranian+churn+dataset")
print("DOI: https://doi.org/10.24432/C5JW3Z")

## 3. EDA

In [ ]:
data_quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
    "unique_count": df.nunique(),
})

display(data_quality)
print(f"Duplicated rows: {df.duplicated().sum():,}")

In [ ]:
display(df.describe().T)

In [ ]:
target_summary = (
    df[target_col]
    .value_counts()
    .rename_axis(target_col)
    .reset_index(name="count")
)
target_summary["percentage"] = target_summary["count"] / len(df) * 100

display(target_summary)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col)
plt.title("Churn Class Distribution")
plt.xlabel("Churn")
plt.ylabel("Customer Count")
plt.show()

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()

plt.figure(figsize=(12, 9))
sns.heatmap(
    df[numeric_columns].corr(),
    cmap="vlag",
    center=0,
    linewidths=0.4,
    annot=False,
)
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
selected_numeric = [
    "call_failure",
    "subscription_length",
    "seconds_of_use",
    "frequency_of_use",
    "frequency_of_sms",
    "customer_value",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for axis, column in zip(axes, selected_numeric):
    sns.boxplot(data=df, x=target_col, y=column, ax=axis)
    axis.set_title(column.replace("_", " ").title())

plt.tight_layout()
plt.show()

In [ ]:
categorical_candidates = ["complains", "age_group", "tariff_plan", "status"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for axis, column in zip(axes, categorical_candidates):
    churn_rate = df.groupby(column)[target_col].mean().reset_index()
    sns.barplot(data=churn_rate, x=column, y=target_col, ax=axis)
    axis.set_title(f"Churn Rate by {column.replace('_', ' ').title()}")
    axis.set_ylabel("Churn Rate")

plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
RAW_FEATURE_COLUMNS = [
    "call_failure",
    "complains",
    "subscription_length",
    "charge_amount",
    "seconds_of_use",
    "frequency_of_use",
    "frequency_of_sms",
    "distinct_called_numbers",
    "age_group",
    "tariff_plan",
    "status",
    "age",
    "customer_value",
]

ENGINEERED_FEATURE_COLUMNS = [
    "usage_minutes",
    "failed_call_rate",
    "sms_share",
    "value_per_usage_minute",
    "value_per_frequency",
    "calls_per_distinct_number",
    "engagement_score",
    "usage_intensity",
]

MODEL_FEATURE_COLUMNS = RAW_FEATURE_COLUMNS + ENGINEERED_FEATURE_COLUMNS
SEGMENT_FEATURES = [
    "customer_value",
    "subscription_length",
    "frequency_of_use",
    "frequency_of_sms",
    "failed_call_rate",
    "complains",
    "usage_minutes",
    "value_per_usage_minute",
]


def ensure_customer_id(data):
    data = data.copy()
    if "customer_id" not in data.columns:
        customer_ids = [f"CUST_{index:04d}" for index in range(1, len(data) + 1)]
        data.insert(0, "customer_id", customer_ids)
    return data


def safe_minmax(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    min_value = series.min()
    max_value = series.max()
    if max_value == min_value:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - min_value) / (max_value - min_value)


def customer_value_thresholds(data):
    values = pd.to_numeric(data["customer_value"], errors="coerce")
    return {
        "low": float(values.quantile(0.33)),
        "high": float(values.quantile(0.67)),
        "min": float(values.min()),
        "max": float(values.max()),
    }


def apply_customer_value_tier(data, thresholds):
    low = float(thresholds.get("low", 0))
    high = float(thresholds.get("high", low))

    def tier(value):
        if value >= high:
            return "High"
        if value >= low:
            return "Medium"
        return "Low"

    return pd.to_numeric(data["customer_value"], errors="coerce").fillna(0).apply(tier)


def add_engineered_features(data, value_thresholds=None):
    data = clean_dataframe_columns(data)
    data = data.copy()

    missing_columns = [column for column in RAW_FEATURE_COLUMNS if column not in data.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    for column in RAW_FEATURE_COLUMNS:
        data[column] = pd.to_numeric(data[column], errors="raise")

    thresholds = value_thresholds or customer_value_thresholds(data)
    data["usage_minutes"] = data["seconds_of_use"] / 60
    data["failed_call_rate"] = data["call_failure"] / (data["frequency_of_use"] + 1)
    data["sms_share"] = data["frequency_of_sms"] / (
        data["frequency_of_sms"] + data["frequency_of_use"] + 1
    )
    data["value_per_usage_minute"] = data["customer_value"] / (data["usage_minutes"] + 1)
    data["value_per_frequency"] = data["customer_value"] / (data["frequency_of_use"] + 1)
    data["calls_per_distinct_number"] = data["frequency_of_use"] / (
        data["distinct_called_numbers"] + 1
    )

    usage_component = safe_minmax(data["frequency_of_use"])
    duration_component = safe_minmax(data["usage_minutes"])
    sms_component = safe_minmax(data["frequency_of_sms"])
    subscription_component = safe_minmax(data["subscription_length"])
    complaint_penalty = np.where(data["complains"] > 0, 0.25, 0)

    data["engagement_score"] = (
        0.35 * usage_component
        + 0.30 * duration_component
        + 0.20 * sms_component
        + 0.15 * subscription_component
        - complaint_penalty
    ).clip(0, 1) * 100
    data["usage_intensity"] = (
        0.60 * safe_minmax(data["frequency_of_use"])
        + 0.40 * safe_minmax(data["usage_minutes"])
    ) * 100
    data["customer_value_tier"] = apply_customer_value_tier(data, thresholds)
    return data, thresholds


data = ensure_customer_id(df)
data, value_thresholds = add_engineered_features(data)
data[target_col] = pd.to_numeric(data[target_col], errors="raise").astype(int)

display(data[["customer_id", *MODEL_FEATURE_COLUMNS, "customer_value_tier", target_col]].head())

## 5. Churn Model Development

In [ ]:
X = data[MODEL_FEATURE_COLUMNS]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(X_train), len(X_test)],
    "churn_rate": [y_train.mean(), y_test.mean()],
})

display(split_summary)

In [ ]:
def available_models():
    models = {
        "logistic_regression": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        max_iter=2_000,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }

    if importlib.util.find_spec("xgboost") is not None:
        from xgboost import XGBClassifier

        models["xgboost"] = XGBClassifier(
            n_estimators=250,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    if importlib.util.find_spec("lightgbm") is not None:
        from lightgbm import LGBMClassifier

        models["lightgbm"] = LGBMClassifier(
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )

    return models


def choose_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    if len(thresholds) == 0:
        return 0.5
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[int(np.nanargmax(f1_values))])


def score_model(model, features, target, threshold):
    probabilities = model.predict_proba(features)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(target, predictions),
        "precision": precision_score(target, predictions, zero_division=0),
        "recall": recall_score(target, predictions, zero_division=0),
        "f1": f1_score(target, predictions, zero_division=0),
        "roc_auc": roc_auc_score(target, probabilities),
        "pr_auc": average_precision_score(target, probabilities),
    }

In [ ]:
results = []
fitted_models = {}
thresholds = {}

for model_name, model in available_models().items():
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    threshold = choose_threshold(y_test, probabilities)
    metrics = score_model(model, X_test, y_test, threshold)

    results.append({"model": model_name, "threshold": threshold, **metrics})
    fitted_models[model_name] = model
    thresholds[model_name] = threshold

model_comparison = pd.DataFrame(results).sort_values(["pr_auc", "f1", "recall"], ascending=False)
display(model_comparison)

In [ ]:
best_model_name = str(model_comparison.iloc[0]["model"])
best_model = fitted_models[best_model_name]
best_threshold = float(thresholds[best_model_name])

test_probabilities = best_model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= best_threshold).astype(int)

print(f"Best model: {best_model_name}")
print(f"Selected threshold: {best_threshold:.4f}")
print(classification_report(y_test, test_predictions, target_names=["not_churn", "churn"]))

confusion = confusion_matrix(y_test, test_predictions)
plt.figure(figsize=(5, 4))
sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["not_churn", "churn"],
    yticklabels=["not_churn", "churn"],
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

risk_driver_thresholds = {
    "failed_call_rate_high": float(data["failed_call_rate"].quantile(0.75)),
    "frequency_of_use_low": float(data["frequency_of_use"].quantile(0.25)),
    "seconds_of_use_low": float(data["seconds_of_use"].quantile(0.25)),
    "customer_value_low": float(data["customer_value"].quantile(0.25)),
}

metadata = {
    "dataset": "Customer Churn.csv",
    "target": target_col,
    "best_model": best_model_name,
    "model_comparison_metrics": model_comparison.to_dict(orient="records"),
    "test_metrics": model_comparison.iloc[0].drop(labels=["model", "threshold"]).to_dict(),
    "raw_feature_columns": RAW_FEATURE_COLUMNS,
    "model_feature_columns": MODEL_FEATURE_COLUMNS,
    "categorical_features": ["complains", "age_group", "tariff_plan", "status"],
    "numeric_features": [
        column
        for column in MODEL_FEATURE_COLUMNS
        if column not in ["complains", "age_group", "tariff_plan", "status"]
    ],
    "threshold": best_threshold,
    "selected_threshold": best_threshold,
    "customer_value_quantile_thresholds": value_thresholds,
    "risk_level_thresholds": {"high": 0.65, "medium": 0.35, "low": 0.0},
    "risk_driver_thresholds": risk_driver_thresholds,
    "training_date": pd.Timestamp.utcnow().isoformat(),
    "dataset_row_count": int(len(data)),
    "churn_rate": float(y.mean()),
}

joblib.dump(best_model, ARTIFACT_DIR / "iranian_churn_model.joblib")
(PROCESSED_DIR / "model_comparison.csv").write_text(model_comparison.to_csv(index=False), encoding="utf-8")
with open(ARTIFACT_DIR / "model_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print(f"Saved model to {ARTIFACT_DIR / 'iranian_churn_model.joblib'}")
print(f"Saved metadata to {ARTIFACT_DIR / 'model_metadata.json'}")

## 6. Customer Segmentation

In [ ]:
def describe_segment(row):
    high_value = row["customer_value"] >= row["global_value_median"]
    high_usage = row["frequency_of_use"] >= row["global_usage_median"]
    high_complaints = row["complains"] >= row["global_complaint_mean"]
    high_failure = row["failed_call_rate"] >= row["global_failure_median"]
    high_churn = row["churn"] >= row["global_churn_mean"]

    if high_value and not high_churn:
        return (
            "High-Value Loyal Customers",
            "Valuable customers with comparatively healthy retention signals.",
            "Maintain loyalty benefits and proactive relationship management.",
        )
    if high_value and (high_churn or high_complaints or high_failure):
        return (
            "At-Risk Heavy Users",
            "Important customers showing service issues or elevated churn behavior.",
            "Prioritize service recovery and personalized retention outreach.",
        )
    if not high_usage:
        return (
            "Low Engagement Customers",
            "Customers with limited usage and weaker day-to-day engagement.",
            "Use re-engagement offers and education campaigns.",
        )
    if not high_value:
        return (
            "Price-Sensitive / Low-Value Users",
            "Lower-value customers who may respond better to low-cost campaigns.",
            "Use automated SMS/email offers and price-sensitive bundles.",
        )
    return (
        "Stable Low-Risk Customers",
        "Customers with steady behavior and no major warning signal.",
        "Monitor routinely and maintain standard lifecycle communications.",
    )


scaler = StandardScaler()
scaled_features = scaler.fit_transform(data[SEGMENT_FEATURES])
segment_model = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=20)
data["segment_id"] = segment_model.fit_predict(scaled_features)

global_values = {
    "global_value_median": data["customer_value"].median(),
    "global_usage_median": data["frequency_of_use"].median(),
    "global_complaint_mean": data["complains"].mean(),
    "global_failure_median": data["failed_call_rate"].median(),
    "global_churn_mean": data["churn"].mean(),
}
segment_profiles = (
    data.assign(**global_values)
    .groupby("segment_id")
    .agg(
        customers=("customer_id", "count"),
        customer_value=("customer_value", "mean"),
        frequency_of_use=("frequency_of_use", "mean"),
        complains=("complains", "mean"),
        failed_call_rate=("failed_call_rate", "mean"),
        churn=("churn", "mean"),
        global_value_median=("global_value_median", "first"),
        global_usage_median=("global_usage_median", "first"),
        global_complaint_mean=("global_complaint_mean", "first"),
        global_failure_median=("global_failure_median", "first"),
        global_churn_mean=("global_churn_mean", "first"),
    )
    .reset_index()
)

label_map = {}
used_names = set()
for _, row in segment_profiles.iterrows():
    name, description, strategy = describe_segment(row)
    if name in used_names:
        name = f"{name} #{int(row['segment_id'])}"
    used_names.add(name)
    label_map[int(row["segment_id"])] = {
        "segment": name,
        "segment_description": description,
        "suggested_strategy": strategy,
    }

data["segment"] = data["segment_id"].map(lambda value: label_map[int(value)]["segment"])
data["segment_description"] = data["segment_id"].map(lambda value: label_map[int(value)]["segment_description"])
data["suggested_strategy"] = data["segment_id"].map(lambda value: label_map[int(value)]["suggested_strategy"])

segment_summary = (
    data.groupby("segment")
    .agg(
        customers=("customer_id", "count"),
        avg_customer_value=("customer_value", "mean"),
        avg_churn_rate=("churn", "mean"),
        complaint_rate=("complains", "mean"),
        suggested_strategy=("suggested_strategy", "first"),
    )
    .reset_index()
    .sort_values("avg_churn_rate", ascending=False)
)

display(segment_summary)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=data,
    x="customer_value",
    y="frequency_of_use",
    hue="segment",
    alpha=0.75,
)
plt.title("Customer Segments by Value and Usage")
plt.xlabel("Customer Value")
plt.ylabel("Frequency of Use")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7. Retention Priority Scoring

In [ ]:
def risk_level_from_probability(churn_probability):
    if churn_probability >= 0.65:
        return "High"
    if churn_probability >= 0.35:
        return "Medium"
    return "Low"


def priority_from_score(score):
    if score >= 80:
        return "P1"
    if score >= 60:
        return "P2"
    if score >= 40:
        return "P3"
    return "P4"


def value_score(customer_value, thresholds, tier):
    min_value = thresholds.get("min")
    max_value = thresholds.get("max")
    if min_value is not None and max_value is not None and max_value > min_value:
        scaled_value = (customer_value - min_value) / (max_value - min_value)
        return float(np.clip(scaled_value * 100, 0, 100))
    return {"Low": 30, "Medium": 65, "High": 100}.get(str(tier), 50)


def service_issue_score(row, failed_call_rate_high):
    score = 0
    if row.get("complains", 0) >= 1:
        score += 60
    if row.get("failed_call_rate", 0) >= failed_call_rate_high:
        score += 40
    return float(np.clip(score, 0, 100))


def explain_customer_risk(row, thresholds):
    reasons = []
    if row.get("complains", 0) >= 1:
        reasons.append("Complaint recorded")
    if row.get("failed_call_rate", 0) >= thresholds["failed_call_rate_high"]:
        reasons.append("High failed call rate")
    if row.get("frequency_of_use", 0) <= thresholds["frequency_of_use_low"]:
        reasons.append("Low usage frequency")
    if row.get("seconds_of_use", 0) <= thresholds["seconds_of_use_low"]:
        reasons.append("Low usage duration")
    if row.get("customer_value", 0) <= thresholds["customer_value_low"]:
        reasons.append("Low customer value")
    if int(row.get("status", 1)) != 1:
        reasons.append("Inactive or higher-risk customer status")
    return "; ".join(reasons[:3]) or "Model detected elevated churn risk based on customer behavior"


def recommended_retention_action(row, thresholds):
    risk_level = row.get("risk_level", "Low")
    value_tier = row.get("customer_value_tier", "Medium")
    has_service_issue = (
        row.get("complains", 0) >= 1
        or row.get("failed_call_rate", 0) >= thresholds["failed_call_rate_high"]
    )

    if risk_level == "High" and value_tier == "High":
        return "Service recovery call" if has_service_issue else "Priority retention call"
    if risk_level == "High" and value_tier == "Medium":
        if row.get("frequency_of_use", 0) <= thresholds["frequency_of_use_low"]:
            return "Re-engagement offer"
        return "Personalized retention offer"
    if risk_level == "High" and value_tier == "Low":
        return "Automated SMS/email campaign"
    if risk_level == "Medium" and value_tier == "High":
        return "Monitor and send loyalty message"
    if risk_level == "Medium":
        return "Low-cost retention campaign"
    return "No immediate action"


full_probabilities = best_model.predict_proba(data[MODEL_FEATURE_COLUMNS])[:, 1]
data["churn_probability"] = full_probabilities
data["churn_prediction"] = (data["churn_probability"] >= best_threshold).astype(int)
data["risk_level"] = data["churn_probability"].apply(risk_level_from_probability)
data["threshold"] = best_threshold

data["main_reason"] = data.apply(
    lambda row: explain_customer_risk(row, risk_driver_thresholds),
    axis=1,
)
data["retention_priority_score"] = data.apply(
    lambda row: int(round(
        0.50 * row["churn_probability"] * 100
        + 0.30 * value_score(row["customer_value"], value_thresholds, row["customer_value_tier"])
        + 0.20 * service_issue_score(row, risk_driver_thresholds["failed_call_rate_high"])
    )),
    axis=1,
)
data["priority"] = data["retention_priority_score"].apply(priority_from_score)
data["recommended_action"] = data.apply(
    lambda row: recommended_retention_action(row, risk_driver_thresholds),
    axis=1,
)

retention_table = data[
    [
        "customer_id",
        "churn_probability",
        "churn_prediction",
        "risk_level",
        "customer_value",
        "customer_value_tier",
        "retention_priority_score",
        "segment",
        "main_reason",
        "recommended_action",
        "priority",
    ]
].sort_values("retention_priority_score", ascending=False)

display(retention_table.head(15))

In [ ]:
segment_output_columns = [
    "customer_id",
    *RAW_FEATURE_COLUMNS,
    "churn",
    *ENGINEERED_FEATURE_COLUMNS,
    "customer_value_tier",
    "segment",
    "segment_description",
    "suggested_strategy",
]
score_output_columns = [
    "customer_id",
    "churn_probability",
    "churn_prediction",
    "risk_level",
    "threshold",
    "customer_value",
    "customer_value_tier",
    "retention_priority_score",
    "segment",
    "main_reason",
    "recommended_action",
    "priority",
]

data[segment_output_columns].to_csv(PROCESSED_DIR / "customer_segments.csv", index=False)
data[score_output_columns].to_csv(PROCESSED_DIR / "customers_scored.csv", index=False)
data[score_output_columns].to_csv(PROCESSED_DIR / "retention_actions.csv", index=False)
joblib.dump({"model": segment_model, "features": SEGMENT_FEATURES, "label_map": label_map}, ARTIFACT_DIR / "segmentation_model.joblib")
joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")

print("Saved processed segmentation and retention outputs.")

## 8. Explainable AI

In [ ]:
permutation = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=15,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": permutation.importances_mean,
        "importance_std": permutation.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

display(importance_df.head(15))

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df.head(15), x="importance_mean", y="feature")
plt.title("Top Permutation Importances")
plt.xlabel("Average Precision Importance")
plt.ylabel("Feature")
plt.show()

In [ ]:
if importlib.util.find_spec("shap") is None:
    print("SHAP is not installed. Permutation importance above is available as model-agnostic XAI.")
else:
    import shap

    sample_features = X_test.sample(min(250, len(X_test)), random_state=RANDOM_STATE)
    try:
        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(sample_features)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        elif np.asarray(shap_values).ndim == 3:
            shap_values = np.asarray(shap_values)[:, :, 1]
        shap.summary_plot(shap_values, sample_features, show=False, max_display=15)
        plt.title("SHAP Summary Plot")
        plt.tight_layout()
        plt.show()
    except Exception as error:
        print(f"SHAP explanation skipped: {error}")